# Dynex SDK - nBit Adder Native Gate Circuit Example

First we import the required packages:

In [1]:
from pennylane import numpy as np
import pennylane as qml
from dynex import DynexConfig, ComputeBackend, DynexCircuit

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model="apollo_rc1", use_notebook_output=True)

We define our circuit:

In [2]:
params = [28938284928182, 15341233312722366482869645212131]  # two numbers to add


def Nqubits(a, b):
    mxVal = a + b
    return mxVal.bit_length()


wires = Nqubits(*params)


def Kfourier(k, wires):
    for j in range(len(wires)):
        qml.RZ(k * np.pi / (2 ** j), wires=wires[j])


def FullAdder(params, state=True):
    a, b = params
    wires = Nqubits(a, b)
    qml.BasisEmbedding(a, wires=range(wires))
    qml.QFT(wires=range(wires))
    Kfourier(b, range(wires))
    qml.adjoint(qml.QFT)(wires=range(wires))
    if state:
        return qml.state()
    else:
        return qml.sample()

We draw the circuit:

In [3]:
# draw circuit:
# too large to draw _ = qml.draw_mpl(FullAdder, style="black_white")(params)
print("Cant't draw circuit with", Nqubits(params[0], params[1]), "wires");

Cant't draw circuit with 104 wires


We execute and measure the circuit on the Dynex platform:

In [4]:
# Execute the circuit on Dynex:
dynex_circuit = DynexCircuit(config=config)
measure = dynex_circuit.execute(FullAdder, params, wires, method="measure",
                                num_reads=10, integration_steps=100)
print("Mesaure:", measure)

INFO: [DYNEX-APOLLO-RC1] Executing PennyLane quantum circuit
INFO: [DYNEX-APOLLO-RC1] Sampler initialised
INFO: [DYNEX-APOLLO-RC1] Apollo QPU chip: apollo_rc1
INFO: [DYNEX-APOLLO-RC1] Settings: num_reads=10, shots=1, annealing_time=100
INFO: [DYNEX-APOLLO-RC1] Submitting the job to Dynex.
INFO: [DYNEX-APOLLO-RC1] SUCCESS: Job created successfully (job_id=7419)
INFO: [DYNEX-APOLLO-RC1] feed_dict: {'cos_rz_0': -0.8534169455411671, 'cos_rz_1': 0.27072407951531846, 'cos_rz_10': 0.3689645437409319, 'cos_rz_100': 0.987167142969934, 'cos_rz_101': -0.996786622846117, 'cos_rz_102': 0.040083520016853794, 'cos_rz_103': -0.7211392098675726, 'cos_rz_11': -0.827334437739942, 'cos_rz_12': -0.29382440526618786, 'cos_rz_13': -0.5942119128449934, 'cos_rz_14': -0.4504376134133375, 'cos_rz_15': -0.5241957585609895, 'cos_rz_16': 0.4877521099077945, 'cos_rz_17': 0.8624824954478191, 'cos_rz_18': 0.9650084184730772, 'cos_rz_19': -0.9912135033566374, 'cos_rz_2': -0.7970960041034325, 'cos_rz_20': -0.06628158357

Mesaure: [1 1 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 0 1 1 0 0 0 1 0 0 1 1 0 1 1
 0 0 1 0 0 0 0 0 1 1 1 0 1 1 1 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 0 0 0 1 1 1 1
 1 0 0 0 0 1 1 0 1 1 0 0 0 1 1 0 0 0 0 1 1 0 1 0 0 1 1 0 0 1]


In [5]:
bitStr = "".join(map(str, measure.astype(int)))
dynexResult = int(bitStr, 2)
print("Dynex Result:", dynexResult)
print("Expected Result:", sum(params))
isValidDynex = dynexResult == sum(params)
print("Is Dynex Result Valid?", isValidDynex)

Dynex Result: 15341233312722366511807930140313
Expected Result: 15341233312722366511807930140313
Is Dynex Result Valid? True
